In [7]:
## Load the environement
from sqlalchemy import over
from dotenv import load_dotenv
load_dotenv(override=True)

True

In [8]:
from helper import snowpark_session


In [9]:
import pandas as pd

snowpark_session.sql("USE WAREHOUSE COMPUTE_WH").collect()
pd.DataFrame(snowpark_session.sql("select * from SNOWFLAKE_LEARNING_DB.CORTEX_AGENTS.SALES_DATA limit 5").collect())

,SALE_ID,CUSTOMER_ID,PRODUCT_ID,SALE_DATE,QUANTITY,UNIT_PRICE,TOTAL_AMOUNT,REGION,SALES_REP
0,1001,1,101,2023-04-15,2,29.99,59.98,North America,John Smith
1,1002,2,102,2023-04-16,1,1299.99,1299.99,Europe,Sarah Johnson
2,1003,3,103,2023-04-17,3,19.99,59.97,Asia,Mike Chen
3,1004,1,104,2023-04-18,1,299.99,299.99,North America,John Smith
4,1005,4,105,2023-04-19,2,49.99,99.98,Europe,Emily Davis


In [10]:
rows = snowpark_session.sql(
    """
    select CONTENT
    from SNOWFLAKE_LEARNING_DB.CORTEX_AGENTS.DOCUMENTS
    limit 1
    """
).collect()

rows

[Row(CONTENT='Our Q1 sales performance exceeded expectations with a 15% increase in revenue compared to last quarter. The North American region showed the strongest growth, followed by Europe and Asia. Key products driving sales include laptops, office furniture, and electronics.')]

In [11]:
# Unstructured data comments
import textwrap
rows = snowpark_session.sql(
    """
    select CONTENT
    from SNOWFLAKE_LEARNING_DB.CORTEX_AGENTS.DOCUMENTS
    limit 1
    """
).collect()
if not rows:
    print("No transcripts found.")
else:
    try:
        transcript = rows[0]['CONTENT']
    except Exception:
        transcript = rows[0][0]
    cleaned = " ".join(transcript.split())
    wrapped = textwrap.fill(cleaned, width=100)

    print("===Meeting Notes ===\n")
    print(wrapped)
    print("\n=== End ===\n")

===Meeting Notes ===

Our Q1 sales performance exceeded expectations with a 15% increase in revenue compared to last
quarter. The North American region showed the strongest growth, followed by Europe and Asia. Key
products driving sales include laptops, office furniture, and electronics.

=== End ===



# Initialize Cortex Agent
This agent will retrieve sales-related data from snowflake using both Text2SQL (done by Cortext Analyst) and Sementic Search(done by Cortex Search).

In [12]:
from snowflake.snowpark import Session
from snowflake.core import Root
from pydantic import BaseModel, PrivateAttr
from snowflake.core.cortex.lite_agent_service import AgentRunRequest
from typing import Any, Type
from langchain.schema import HumanMessage
from langgraph.graph import END
from langgraph.types import Command
from typing import Literal, Dict
import json

from helper import State

In [19]:
SEMANTIC_MODEL_FILE = "@SNOWFLAKE_LEARNING_DB.MODELS.SEMANTIC_MODELS/semantic_models/business_analytics_model.yaml"
CORTEX_SEARCH_SERVICE = "SNOWFLAKE_LEARNING_DB.CORTEX_AGENTS.SALES_CONTENT_SEARCH"

In [14]:
class CortexAgentArgs(BaseModel):
    query: str

In [21]:
class CortexAgentTool:
    name: str = "CortexAgent"
    description: str = "answers questions using sales conversations and metrics"
    args_schema: Type[CortexAgentArgs] = CortexAgentArgs
    _session: Session = PrivateAttr()
    _root: Root = PrivateAttr()
    _agent_service: Any = PrivateAttr()

    def __init__(self, session:Session):
        self._session = session
        self._root = Root(session)
        self._agent_service = self._root.cortex_agent_service

    def _build_request(self, query: str) -> AgentRunRequest:
        return AgentRunRequest.from_dict({
            "model": "claude-3-5-sonnet",
            "tools":[
                {"tool_spec": {"type": "cortex_analyst_text_to_sql", "name": "analyst1"}},
                {"tool_spec": {"type": "cortex_search", "name": "search1"}},
            ],
            "tool_resources": {
                "analyst1": {"semantic_model_file": SEMANTIC_MODEL_FILE},
                "search1": {
                    "name": CORTEX_SEARCH_SERVICE,
                    "max_results": 10,
                    "id_column": "conversation_id"
                }
            },
            "messages":[
                {"role": "user", "content": [{"type": "text", "text": query}]}
            ]
        })

    def _consume_stream(self, stream):
        text, sql, citations = "", "", []
        for evt in stream.events():
            try:
                delta = (evt.data.get("delta") if isinstance(evt.data, dict) else json.loads(evt.data).get("delta") or json.loads(evt.data).get("data", {}).get("delta"))
            except Exception:
                continue

            if not isinstance(delta, dict):
                continue

            for item in delta.get("context", []):
                if item.get("type") == 'text':
                    text += item.get("text", "")
                elif item.get("type") == "tool_results":
                    for result in item.get("tool_results").get("content", []):
                        if result.get("type") != "json":
                            continue
                        j = result["json"]
                        text += j.get("text", "")
                        sql = j.get("sql", sql)
                        citations.extend({
                            "source_id": s.get("source_id"),
                            "doc_id": s.get("doc_id")
                        } for s in j.get("searchResults", []))
        return text, sql, citations


    def run (self, query: str, **kwargs):
        """
        This agent will retrieve sales-related data from Snowflake using both Text2SQL and Semantic Search.
        """
        req = self._build_request(query)
        stream = self._agent_service.run(req)
        text, sql, citations = self._consume_stream(stream)

        results_str = ""
        if sql:
            try:
                self._session.sql("LIST @SNOWFLAKE_LEARNING_DB.MODELS.SEMANTIC_MODELS").show()
                self._session.sql("USE DATABASE SNOWFLAKE_LEARNING_DB").collect()
                self._session.sql("USE SCHEMA MODELS").collect()
                self._session.sql("USE WAREHOUSE COMPUTE_WH").collect()
                df = self._session.sql(sql.rstrip(";")).to_pandas()
                results_str = df.to_string(index=False)
            except Exception as e:
                results_str = f"SQL execution error: {e}"
        return text, citations, sql, results_str


In [22]:
cortex_agnet_tool = CortexAgentTool(session=snowpark_session)

In [23]:
from langgraph.prebuilt import create_react_agent
from helper import agent_system_prompt
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o")

cortex_agent = create_react_agent(
    llm,
    tools=[cortex_agnet_tool.run],
    prompt = agent_system_prompt(f"""
        You are the Researcher. You can answer questions using customer deal data along with meeting notes. Do not take any further action.
    """)
)

In [24]:
agent_response = cortex_agent.invoke({"messages": "What are our top 3 customer deals?"})
print(agent_response['messages'][-1].content)

It seems there is an issue with accessing the required database for querying the top 3 customer deals. The error indicates a missing file or misconfiguration. Since I'm limited in handling this type of error directly, please let the technical team know so they can resolve the configuration issue.


In [ ]:
agent_response = cortex_agent.invoke({"messaeges": "what is the next step in the sales process for healthtech?"})
print(agent_response['messages'][-1].content)

I'm unable to access the deal data or meeting notes for "Capital Corp Collaboration" due to authorization issues with the search service. You might need someone with the appropriate access permissions to retrieve this information.


In [ ]:
def cortex_agents_research_node(
    state: State,
) -> Command[Literal["executor"]]:
    query = state.get("agent_query", state.get("user_query", ""))
    # Call the tool with the string query
    agent_response = cortex_agent.invoke({"messages":query})
    # Compose a message content string with all results new HumanMessage with the result
    new_message = HumanMessage(content=agent_response['messages'][-1].content, name="cortex_researcher")
    # Append to the message history
    goto = "executor"
    return Command(
        update={"messages": [new_message]},
        goto=goto,
    )

In [ ]:
from helper import State, planner_node, executor_node, web_research_node, chart_node, chart_summary_node, synthesizer_node

from langgraph.graph import START, StateGraph

workflow = StateGraph(State)
workflow.add_node("planner", planner_node)
workflow.add_node("executor", executor_node)
workflow.add_node("web_researcher", web_research_node)
workflow.add_node("cortex_researcher", cortex_agents_research_node)
workflow.add_node("chart_generator", chart_node)
workflow.add_node("chart_summarizer", chart_summary_node)
workflow.add_node("synthesizer", synthesizer_node)

workflow.add_edge(START, "planner")

graph = workflow.compile()

In [ ]:
from langchain.schema import HumanMessage

query = "What are our top 3 customer deals? Chart the deal value for each."
print(f"Query: {query}")
state = {
            "messages": [HumanMessage(content=query)],
            "user_query": query,
            "enabled_agents": ["cortex_researcher", "web_researcher", "chart_generator", "chart_summarizer", "synthesizer"],
        }
graph.invoke(state)

print("--------------------------------")

Query: What are our top 3 customer deals? Chart the deal value for each.


KeyboardInterrupt: 

In [ ]:
query = "Identify our pending deals, research if they may be experiencing regulatory changes, and using the meeting notes for each customer, provide a new value proposition for each given the regulatory changes."
print(f"Query: {query}")

state = {
            "messages": [HumanMessage(content=query)],
            "user_query": query,
            "enabled_agents": ["cortex_researcher", "web_researcher", "chart_generator", "chart_summarizer", "synthesizer"],
        }
graph.invoke(state)

print("--------------------------------")

Query: Identify our pending deals, research if they may be experiencing regulatory changes, and using the meeting notes for each customer, provide a new value proposition for each given the regulatory changes.


KeyboardInterrupt: 

In [ ]:
query = "Is there a common theme across our meeting notes?"
print(f"Query: {query}")

state = {
            "messages": [HumanMessage(content=query)],
            "user_query": query,
            "enabled_agents": ["cortex_researcher", "web_researcher", "chart_generator", "chart_summarizer", "synthesizer"],
        }
graph.invoke(state)

print("--------------------------------")